## 3. Análisis Exploratorio de Datos (EDA)

El objetivo de este análisis es responder las siguientes preguntas
sobre el comportamiento histórico de Bitcoin:

- ¿Cómo evolucionó el precio a lo largo de los años?
- ¿Cuáles fueron los períodos de mayor volatilidad?
- ¿Cuál fue el rendimiento anual?
- ¿Qué relación existe entre el volumen de operaciones y el movimiento del precio?
- ¿En qué meses o años el mercado fue más estable o más volátil?


### 3.1 Evolución del precio a lo largo de los años

Se analizó el precio de cierre diario de Bitcoin desde 2015 hasta la actualidad.
El objetivo es identificar tendencias generales, ciclos de crecimiento y caídas
significativas a lo largo del tiempo.

In [2]:
import pandas as pd


btc = pd.read_csv("../data/bitcoin_limpio.csv", parse_dates=['Date'])

#### **Primero — estadísticas generales**

In [3]:
print("ESTADÍSTICAS GENERALES")
print(f"Precio promedio historico: {btc["Close"].mean():.2f} USD")
print(f"Precio máximo historico: {btc["Close"].max():.2f} USD")
print(f"Precio mínimo historico: {btc["Close"].min():.2f} USD")

ESTADÍSTICAS GENERALES
Precio promedio historico: 29385.92 USD
Precio máximo historico: 124752.53 USD
Precio mínimo historico: 178.10 USD


#### **Segundo — estadísticas por año**

In [4]:
btc["Year"] = btc["Date"].dt.year

print("\nESTADÍSTICAS POR AÑO")
resumen_anual = btc.groupby("Year")["Close"].agg(
    promedio="mean",
    maximo="max",
    minimo="min"
).round(2)
print(resumen_anual)


ESTADÍSTICAS POR AÑO
       promedio     maximo    minimo
Year                                
2015     272.45     465.32    178.10
2016     568.49     975.92    364.33
2017    4006.03   19497.40    777.76
2018    7572.30   17527.00   3236.76
2019    7395.25   13016.23   3399.47
2020   11116.38   29001.72   4970.79
2021   47436.93   67566.83  29374.15
2022   28197.75   47686.81  15787.28
2023   28859.45   44166.60  16625.08
2024   65964.12  106140.60  39507.37
2025  101641.92  124752.53  76271.95
2026   75564.77   96929.33  60867.41


#### Conclusión
En 2015 el precio promedio de Bitcoin era de 272 USD, mientras que en 2025 
alcanzó los 101.641 USD, lo que representa un crecimiento extraordinario 
a lo largo de la década.

El año que registró el mayor incremento fue 2021, cuando el precio promedio 
pasó de 11.116 USD a 47.436 USD, cuadruplicando su valor respecto al año anterior.

Sin embargo, no todos los años fueron positivos. En 2022 y 2023 el precio 
promedio cayó respecto a 2021, manteniéndose por debajo de los 29.000 USD, 
lo que refleja la alta ciclicidad que caracteriza al mercado de Bitcoin.

### 3.2 Períodos de mayor volatilidad

Se calculó la volatilidad como la desviación estándar de los rendimientos diarios
en una ventana móvil de 30 días. El objetivo es identificar en qué momentos
el mercado tuvo movimientos más bruscos e impredecibles.

In [5]:
btc["rendimiento_diario"] = btc["Close"].pct_change()

btc["volatilidad_30d"] = btc["rendimiento_diario"].rolling(30).std() 

print("\nESTADÍSTICAS DE RENDIMIENTO Y VOLATILIDAD\n")
print(btc[['Date', 'Close', 'rendimiento_diario', 'volatilidad_30d']].head(10))


ESTADÍSTICAS DE RENDIMIENTO Y VOLATILIDAD

        Date       Close  rendimiento_diario  volatilidad_30d
0 2015-01-01  314.248993                 NaN              NaN
1 2015-01-02  315.032013            0.002492              NaN
2 2015-01-03  281.082001           -0.107767              NaN
3 2015-01-04  264.195007           -0.060079              NaN
4 2015-01-05  274.473999            0.038907              NaN
5 2015-01-06  286.188995            0.042682              NaN
6 2015-01-07  294.337006            0.028471              NaN
7 2015-01-08  283.348999           -0.037331              NaN
8 2015-01-09  290.407990            0.024913              NaN
9 2015-01-10  274.795990           -0.053759              NaN


In [6]:
top_volatilidad = btc.sort_values("volatilidad_30d", ascending=False).head(10)

print("TOP 10 PERÍODOS MÁS VOLÁTILES\n")
print(top_volatilidad[['Date', "Open",'Close', 'volatilidad_30d']])

TOP 10 PERÍODOS MÁS VOLÁTILES

           Date          Open         Close  volatilidad_30d
1922 2020-04-06   6788.049805   7271.781250         0.091330
1919 2020-04-03   6797.396484   6733.387207         0.090578
1918 2020-04-02   6606.776367   6793.624512         0.090573
1926 2020-04-10   7303.815430   6865.493164         0.090525
1916 2020-03-31   6430.606445   6438.644531         0.090507
1915 2020-03-30   5925.538574   6429.841797         0.090499
1917 2020-04-01   6437.319336   6606.776367         0.090373
1920 2020-04-04   6738.382812   6867.527344         0.090365
1921 2020-04-05   6862.537598   6791.129395         0.090355
1097 2018-01-02  13625.000000  14982.099609         0.090238


In [7]:
volatilidad_anual = btc.groupby("Year")["volatilidad_30d"].mean().round(4) * 100

print("\nVOLATILIDAD PROMEDIO ANUAL\n")
print(volatilidad_anual)


VOLATILIDAD PROMEDIO ANUAL

Year
2015    3.01
2016    2.24
2017    4.46
2018    4.08
2019    3.44
2020    3.30
2021    4.09
2022    3.26
2023    2.19
2024    2.71
2025    2.15
2026    2.40
Name: volatilidad_30d, dtype: float64


#### Conclusión
Los años más volátiles fueron 2017, 2018 y 2021, todos con más del 4% 
de volatilidad diaria promedio. No es casualidad: estos mismos años 
fueron los de mayores movimientos de precio, donde Bitcoin llegó a 
cuadruplicar su valor en 2021 respecto al año anterior.

Por otro lado, los años más estables fueron 2016, 2023 y 2025, 
con volatilidades entre 2% y 2.5%. En estos períodos el precio 
no mostró subidas ni bajadas muy marcadas respecto a los años anteriores.

Esto sugiere una relación directa entre volatilidad y movimiento de precio:
los años de mayor crecimiento o caída son también los años de mayor volatilidad.

### 3.3 Rendimiento anual

Se calculó el rendimiento anual de Bitcoin comparando el precio de cierre
del primer y último día de cada año. El objetivo es entender cuánto
creció o cayó el precio en cada año calendario.

In [8]:
rendimiento_anual = btc.groupby("Year")["Close"].agg(
    inicio="first",
    fin="last"
)

rendimiento_anual["rendimiento"] = ((rendimiento_anual["fin"] - rendimiento_anual["inicio"]) / rendimiento_anual["inicio"] * 100).round(2)

print("\nRENDIMIENTO ANUAL DE BITCOIN\n")

print(rendimiento_anual[["inicio", "fin", "rendimiento"]])


RENDIMIENTO ANUAL DE BITCOIN

            inicio           fin  rendimiento
Year                                         
2015    314.248993    430.566986        37.01
2016    434.334015    963.742981       121.89
2017    998.325012  14156.400391      1318.02
2018  13657.200195   3742.700439       -72.60
2019   3843.520020   7193.599121        87.16
2020   7200.174316  29001.720703       302.79
2021  29374.152344  46306.445312        57.64
2022  47686.812500  16547.496094       -65.30
2023  16625.080078  42265.187500       154.23
2024  44167.332031  93429.203125       111.53
2025  94419.757812  87508.828125        -7.32
2026  88731.984375  61695.000000       -30.47


#### Conclusión
El mejor año en términos de rendimiento fue 2017 con un 1318%, 
impulsado por la gran burbuja especulativa de ese período.

Los peores años fueron 2018 y 2022 con rendimientos negativos 
de -72% y -65% respectivamente, ambos años inmediatamente posteriores 
a períodos de gran crecimiento.

Esto sugiere un patrón cíclico en Bitcoin: años de rendimientos 
extraordinarios son seguidos por años de fuertes caídas. Se puede 
observar este ciclo en 2017-2018, 2020-2021-2022, y 2023-2024-2025.

### 3.4 Relación entre volumen de operaciones y movimiento del precio

Se analizó la relación entre el volumen operado diario y los movimientos
del precio de cierre. El objetivo es identificar si los días de mayor
actividad del mercado coinciden con movimientos significativos en el precio.

In [9]:
volumne_anual = btc.groupby("Year")["Volume"].mean().round(2)

print("\nVOLUMEN PROMEDIO ANUAL\n")
print(volumne_anual)


VOLUMEN PROMEDIO ANUAL

Year
2015    3.390557e+07
2016    8.592451e+07
2017    2.382867e+09
2018    6.063552e+09
2019    1.673049e+10
2020    3.302327e+10
2021    4.715574e+10
2022    3.001327e+10
2023    1.825093e+10
2024    3.743909e+10
2025    5.301822e+10
2026    3.994794e+10
Name: Volume, dtype: float64


In [10]:
correlacion = btc["Volume"].corr(btc["rendimiento_diario"])

print(f"\nCORRELACIÓN ENTRE VOLUMEN Y RENDIMIENTO DIARIO: {correlacion:.4f}")


CORRELACIÓN ENTRE VOLUMEN Y RENDIMIENTO DIARIO: -0.0222


#### Conclusión
El volumen operado creció de forma sostenida a lo largo de los años,
pasando de un promedio de 33 millones de dólares diarios en 2015 
a más de 53 mil millones en 2025. Esto refleja una mayor adopción 
y maduración del mercado de Bitcoin.

Sin embargo, la correlación entre volumen y rendimiento diario 
fue de -0.02, un valor prácticamente nulo. Esto indica que el 
volumen operado no es un indicador del movimiento del precio 
en el corto plazo.

### 3.5 Estabilidad y volatilidad por mes y año

Se agruparon los datos por mes y por año para calcular la volatilidad
promedio en cada período. El objetivo es identificar patrones estacionales
y determinar qué períodos históricamente esta más tranquilos o más agitado.

In [11]:
btc["Mes"] = btc["Date"].dt.month

volatilidad_mensualk = btc.groupby("Mes")["volatilidad_30d"].mean().round(4) * 100

print("\nVOLATILIDAD PROMEDIO MENSUAL\n")
print(volatilidad_mensualk)


VOLATILIDAD PROMEDIO MENSUAL

Mes
1     3.46
2     3.59
3     3.70
4     3.15
5     2.87
6     3.23
7     3.23
8     3.07
9     2.76
10    2.32
11    3.05
12    3.30
Name: volatilidad_30d, dtype: float64


#### Conclusión
Los meses con mayor volatilidad histórica fueron marzo (3.70%), 
febrero (3.59%) y enero (3.46%), sugiriendo que el inicio del año 
tiende a ser el período más agitado para Bitcoin.

Por el contrario, los meses más estables fueron octubre (2.32%), 
septiembre (2.76%) y agosto (3.07%), indicando que la segunda mitad 
del año tiende a ser más tranquila en términos de movimiento de precio.

Esto podría reflejar patrones estacionales en el comportamiento 
de los inversores, aunque se requeriría un análisis más profundo 
para confirmar esta hipótesis.